## 📌 이 실습 노트북 사용법

이 노트북은 `1. Data Preprocessing.ipynb`의 내용을 바탕으로, **여러분이 직접 코드를 작성해보는 실습용 노트북**입니다.

- 마크다운(설명) 셀은 그대로 남아있고, 그 아래 코드 셀은 **비어 있습니다**.
- 앞서 작성한 노트북의 코드를 참고해도 되고, **AI(예: GitHub Copilot Chat, ChatGPT, Claude 등)에게 무엇을 만들어야 하는지 설명하고, AI가 작성해준 코드를 코드 셀에 채워 넣어도 됩니다.**
- 각 코드 셀 위의 마크다운에는 **🎯 실습 과제** 영역이 있습니다. 여기에는 다음 내용이 들어있습니다.
  - **목표** — 무엇을 만들어야 하는지
  - **꼭 사용해야 하는 변수/함수 이름** — 뒤 단계의 코드가 이 이름을 그대로 사용하기 때문에, AI에게 지시할 때 이 이름을 반드시 알려줘야 합니다.
  - **알아두면 좋은 용어** — AI에게 지시할 때 사용할 수 있는 핵심 키워드
  - **AI에게 줄 수 있는 프롬프트 예시**

### 💡 AI에게 잘 지시하는 방법 (Tip)

좋은 프롬프트에는 보통 아래 4가지가 들어갑니다.

1. **입력(Input)** — 어떤 변수/데이터를 사용할지 (예: `signal`, shape, 자료형)
2. **처리 과정(Process)** — 어떤 작업을 할지 (예: "FFT를 적용해서...")
3. **출력(Output)** — 결과를 어떤 변수명에 저장할지, shape나 형태
4. **확인(Check)** — 결과를 어떻게 확인할지 (예: "shape를 출력해줘", "그래프로 그려줘")

> 예: "`signal`이라는 1차원 numpy 배열(길이 486224)이 있어. 이 배열을 `window_size=2048`, `step=1024`로 슬라이딩 윈도우 슬라이싱해서 `ir_segments`라는 2차원 배열로 만들어줘. 결과 shape를 출력해줘."

⚠️ **변수명/함수명을 마크다운에 적힌 그대로 사용하세요.** 뒤 단계의 실습 과제 설명은 앞 단계에서 만든 변수명을 그대로 가정하고 작성되어 있습니다.

---

---
## STEP 1. 데이터 불러오기

CWRU 베어링 데이터셋은 `.mat` 파일(매트랩 데이터 형식)로 제공됩니다. 각 파일에는 `X{번호}_DE_time`이라는 이름으로 **Drive End(모터 구동축) 가속도계 진동 신호**가 시간 순서대로 저장되어 있습니다. (예: `IR007_1_110.mat` 파일에는 `X110_DE_time`)

### 🎯 실습 과제

**목표**

`dataset/IR007_1_110.mat` 파일을 불러와서, 그 안에 들어있는 DE 채널 진동 신호를 1차원 배열로 추출합니다.

**알아두면 좋은 용어**

- `.mat` 파일 / 딕셔너리(dict) 형태
- 1차원 배열 vs 2차원 배열, `shape`

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | 파일 경로 | `'dataset/IR007_1_110.mat'` |
| 출력 | `signal` | 1차원 numpy 배열, shape `(486224,)` |

**AI에게 줄 수 있는 프롬프트 예시**

> ".mat 파일을 불러와서, 그 안의 DE 채널 진동 신호를 shape (486224,)인 1차원 numpy 배열 signal로 만들어줘."


---
## STEP 2. Train / Test 분리

딥러닝 모델을 만들 때는 데이터를 **Train(훈련)**과 **Test(평가)**로 나눠야 합니다. Train 데이터로 모델을 학습시키고, 한 번도 보여주지 않은 Test 데이터로 "새로운 데이터에도 잘 작동하는지"를 확인합니다.

⚠️ 이번 신호는 **시간 순서가 있는 데이터**이기 때문에, 무작위로 섞어서 나누면 안 됩니다. 뒤섞어서 나누면 Train과 Test에 거의 똑같은 구간이 섞여 들어가서 (Data Leakage), 평가 결과를 믿을 수 없게 됩니다. 그래서 **신호를 시간 순서대로 앞부분(Train) / 뒷부분(Test)으로 먼저 잘라**둡니다.

### 🎯 실습 과제

**목표**

`signal`을 시간 순서대로 앞 80% / 뒤 20%로 나눠서 `train_signal`, `test_signal`을 만듭니다.

**알아두면 좋은 용어**

- train / test split, 비율(ratio)
- Data Leakage (데이터가 새는 문제)

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | `signal` | 1차원 numpy 배열, shape `(486224,)` |
| 출력 | `train_signal` | 1차원 numpy 배열, shape 약 `(388979,)` (전체의 80%) |
| 출력 | `test_signal` | 1차원 numpy 배열, shape 약 `(97245,)` (전체의 20%) |

**AI에게 줄 수 있는 프롬프트 예시**

> "signal(shape (486224,))을 시간 순서대로 앞 80% / 뒤 20%로 나눠서 train_signal, test_signal을 만들어줘. 두 배열의 shape를 출력해줘."


---
## STEP 3. 윈도우 슬라이싱 (Sliding Window)

48만 개짜리 신호를 통째로 모델에 넣을 수는 없습니다. 딥러닝 모델은 **고정된 크기의 입력**만 받을 수 있기 때문입니다.

그래서 신호를 일정한 크기(`window_size`)로 잘라서 여러 개의 학습 샘플로 만듭니다.

```
[■■■■■□□□□□□□□□□]  ← 1번째 샘플 (0 ~ 2047)
[□■■■■■□□□□□□□□□]  ← 2번째 샘플 (1024 ~ 3071, step만큼 이동)
[□□■■■■■□□□□□□□□]  ← 3번째 샘플 (2048 ~ 4095)
```

- `window_size` : 한 번에 잘라낼 길이 (여기서는 2048개)
- `step` : 다음 샘플을 만들 때 몇 개씩 이동할지 (여기서는 1024개, 즉 50% 겹침/overlap)

💡 **`window_size`는 어떻게 정할까?**

베어링이 손상되면 축이 한 바퀴 돌 때마다 "딱" 하는 충격이 반복됩니다. `window_size`가 축 1바퀴보다 짧으면, 어떤 윈도우는 이 충격을 하나도 포함하지 못해 정상/결함을 구분할 수 없습니다. 그래서 "축 1바퀴 이상의 길이"를 고르고, 계산 효율을 위해 보통 2의 거듭제곱(512, 1024, 2048, 4096 ...)으로 맞춥니다.

💡 **왜 겹치게(overlap) 만들까?**

`step`을 `window_size`보다 작게 잡으면, 1) 같은 신호에서 더 많은 학습 샘플을 얻을 수 있고(데이터 증강), 2) 결함 충격 패턴이 한 윈도우의 경계에 걸려 잘리더라도 겹치는 다음 윈도우에는 온전히 포함될 가능성이 높아집니다.

### 🎯 실습 과제

**목표**

`train_signal`, `test_signal`을 각각 `window_size=2048`, `step=1024`로 슬라이딩 윈도우 슬라이싱해서 2차원 배열로 변환합니다.

**알아두면 좋은 용어**

- window_size, step(stride)
- overlap(겹침)
- 2차원 배열의 shape = (조각 개수, window_size)

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | `train_signal` | 1차원 numpy 배열, shape `(388979,)` |
| 입력 | `test_signal` | 1차원 numpy 배열, shape `(97245,)` |
| 출력 | `train_segments` | 2차원 numpy 배열, shape 약 `(378, 2048)` |
| 출력 | `test_segments` | 2차원 numpy 배열, shape 약 `(93, 2048)` |

**AI에게 줄 수 있는 프롬프트 예시**

> "train_signal과 test_signal(1차원 배열)을 각각 window_size=2048, step=1024로 슬라이딩 윈도우 방식으로 잘라서 2차원 배열 train_segments, test_segments로 만들어줘 (shape: (조각 개수, window_size)). 각각의 shape를 출력해줘."


---
## STEP 4. FFT 변환 - 시간 정보를 주파수 정보로

**FFT (Fast Fourier Transform)** 는 복잡한 진동 신호를 "어떤 주기로 진동하는 성분이 얼마나 섞여 있는가"로 분해해주는 변환입니다.

- 피아노로 "도 + 미 + 솔" 화음을 동시에 쳤을 때, FFT는 "이 소리는 도, 미, 솔 세 음이 섞여있다"고 알려주는 것과 비슷합니다.
- 베어링이 손상되면 특정 주파수에서 진동이 유독 강하게 나타나는데, 시간 영역(time domain)에서는 잘 안 보이던 패턴이 주파수 영역(frequency domain)에서는 뚜렷하게 보일 수 있습니다.

파이썬에선 이 복잡한 FFT를 직접 구현할 필요 없이, 이미 구현되어 있는 함수를 사용하면 됩니다.

### 🎯 실습 과제

**목표**

`train_segments`, `test_segments`의 각 윈도우(행)에 FFT를 적용해서, 주파수 영역의 진폭(magnitude) 값으로 변환합니다. (FFT 결과의 앞쪽 절반, 길이 `window_size // 2`만 사용합니다.)

**알아두면 좋은 용어**

- FFT(고속 푸리에 변환), 시간 영역 / 주파수 영역
- 진폭(magnitude)

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | `train_segments` | 2차원 numpy 배열, shape `(378, 2048)` |
| 입력 | `test_segments` | 2차원 numpy 배열, shape `(93, 2048)` |
| 출력 | `train_data` | 2차원 numpy 배열, shape `(378, 1024)` |
| 출력 | `test_data` | 2차원 numpy 배열, shape `(93, 1024)` |

**AI에게 줄 수 있는 프롬프트 예시**

> "train_segments와 test_segments(2차원 배열, 각 행이 길이 2048인 시간 영역 신호)의 각 행에 FFT를 적용해서, 길이 1024의 진폭(magnitude) 값으로 변환한 2차원 배열 train_data, test_data를 만들어줘. shape를 출력해줘."


---
## STEP 5. 정규화 (Normalization)

딥러닝 모델은 입력값의 **크기(스케일)** 차이에 민감합니다. 수치가 큰 데이터에 크게 반응하는 경향이 있어서, 값의 편차가 클 경우 수치가 작은 데이터들을 무시하게 될 우려가 있습니다.

FFT 결과값은 주파수마다 크기가 크게 차이날 수 있으므로, **평균 0, 표준편차 1**로 맞춰줍니다.

⚠️ **Data Leakage 방지**

정규화 기준(평균/표준편차)은 **train 데이터에서만** 계산(`fit`)하고, test 데이터에는 그 기준으로 **변환만(`transform`)** 적용해야 합니다. 전체 데이터(train+test)로 먼저 통계를 계산하면, test 데이터의 정보가 학습 과정에 "새어 들어가는"(Data Leakage) 문제가 생깁니다.

### 🎯 실습 과제

**목표**

`train_data`로 정규화 기준(평균/표준편차)을 계산하고, `train_data`와 `test_data`를 모두 그 기준으로 정규화합니다.

**알아두면 좋은 용어**

- 정규화(normalization), 평균/표준편차
- fit vs transform
- Data Leakage

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | `train_data` | 2차원 numpy 배열, shape `(378, 1024)` |
| 입력 | `test_data` | 2차원 numpy 배열, shape `(93, 1024)` |
| 출력 | `train_scaled` | 2차원 numpy 배열, `train_data`와 동일한 shape |
| 출력 | `test_scaled` | 2차원 numpy 배열, `test_data`와 동일한 shape |

**AI에게 줄 수 있는 프롬프트 예시**

> "train_data로 정규화 기준(평균/표준편차)을 학습(fit)시키고, train_data와 test_data를 둘 다 그 기준으로 정규화(transform)해서 train_scaled, test_scaled를 만들어줘. (test_data로는 fit하지 마.) 정규화 전/후 train_data와 train_scaled의 평균과 표준편차를 출력해줘."


---
## STEP 6. 지금까지의 과정을 함수로 묶기

지금까지 파일 1개(`IR007_1_110.mat`)를 가지고 STEP 1~5의 전처리 과정을 직접 만들어봤습니다.

다음 단계에서는 같은 과정을 10개 파일 전체에 반복해야 합니다. 매번 같은 코드를 다시 작성하지 않도록, STEP 1~5 과정을 **하나의 함수**로 묶어둡니다.

### 🎯 실습 과제

**목표**

`.mat` 파일 경로 하나를 입력받아서, STEP 1(불러오기) ~ STEP 5(정규화)까지의 과정을 모두 수행하고 `(X_train, X_test)`를 반환하는 함수를 만듭니다.

**알아두면 좋은 용어**

- 함수(function), 매개변수(입력)/반환값(출력)
- 재사용성(reusability)

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | `file_path` | 문자열, 예: `'dataset/IR007_1_110.mat'` |
| 출력 | `X_train` | 2차원 numpy 배열, shape `(378, 1024)` |
| 출력 | `X_test` | 2차원 numpy 배열, shape `(93, 1024)` |

**AI에게 줄 수 있는 프롬프트 예시**

> "위에서 작업한 STEP 1~5 과정을 모두 합쳐서, .mat 파일 경로(file_path)를 입력받아 (X_train, X_test)를 반환하는 함수 preprocess_file(file_path)을 만들어줘. 이 함수로 'dataset/IR007_1_110.mat'을 처리한 결과의 shape를 출력해서 확인해줘."


---
## STEP 7. 10개 파일 전체로 확장 — 데이터셋 완성

이제 STEP 6에서 만든 함수를 10개 파일 전체에 적용합니다. 각 파일은 4가지 클래스(Normal / Inner Race / Outer Race / Ball) 중 하나에 속하므로, 파일마다 **정답 라벨(label)**도 함께 붙여줘야 합니다.

| 파일명 | 클래스 (레이블) |
|---|---|
| Time_Normal_1_098.mat | Normal (0) |
| IR007_1_110.mat / IR014_1_175.mat / IR021_1_214.mat | Inner Race (1) |
| OR007_6_1_136.mat / OR014_6_1_202.mat / OR021_6_1_239.mat | Outer Race (2) |
| B007_1_123.mat / B014_1_190.mat / B021_1_227.mat | Ball (3) |

각 파일을 함수에 통과시키면 `(X_train, X_test)` 조각이 나옵니다. 이 조각들을 모두 이어붙이고(concatenate), 같은 길이의 라벨 배열도 함께 만들어서 최종 데이터셋을 완성합니다.

### 🎯 실습 과제

**목표**

10개 파일 각각에 STEP 6의 함수를 적용한 뒤, 모든 결과를 하나로 합쳐서 최종 `X_train`, `X_test`, `y_train`, `y_test`를 만듭니다.

**알아두면 좋은 용어**

- 라벨(label) / 클래스(class)
- concatenate (배열 이어붙이기)
- X(입력 데이터) / y(정답 라벨)

**입력 / 출력**

| | 이름 | 형태 |
|---|---|---|
| 입력 | 파일 경로 10개 + 각 파일의 클래스 번호(0~3) | - |
| 출력 | `X_train` | 2차원 numpy 배열, shape `(전체 train 조각 수, 1024)` |
| 출력 | `X_test` | 2차원 numpy 배열, shape `(전체 test 조각 수, 1024)` |
| 출력 | `y_train` | 1차원 numpy 배열, `X_train`과 같은 길이, 값은 0~3 |
| 출력 | `y_test` | 1차원 numpy 배열, `X_test`와 같은 길이, 값은 0~3 |

**AI에게 줄 수 있는 프롬프트 예시**

> "10개의 (파일 경로, 클래스 번호) 쌍의 목록을 만들어줘 (Normal=0, Inner Race=1, Outer Race=2, Ball=3). 각 파일에 preprocess_file 함수를 적용해서 나온 X_train, X_test 조각들을 모두 concatenate해서 합치고, 같은 길이의 클래스 라벨 배열도 만들어서 y_train, y_test를 만들어줘. 최종 X_train, X_test, y_train, y_test의 shape를 출력해줘."


#### 📋 전체 흐름 정리

| 단계 | 내용 | 핵심 포인트 |
|---|---|---|
| **STEP 1. 데이터 불러오기** | `.mat` 파일에서 진동 신호(시계열 데이터) 읽기 | `.mat` 파일은 dict, DE 채널 신호를 1차원 배열로 |
| **STEP 2. Train / Test 분리** | 신호를 시간 순서대로 앞부분(train)·뒷부분(test)으로 분리 | 시간 순서가 있는 데이터는 무작위로 섞으면 안 됨 (Data Leakage) |
| **STEP 3. 윈도우 슬라이딩** | 긴 신호를 일정한 크기로 잘라서 여러 개의 샘플로 변환 | window_size / step, overlap의 의미 |
| **STEP 4. FFT 적용** | 시간 영역 신호 → 주파수 영역 신호로 변환 | 패턴을 더 쉽게 구분할 수 있게 됨 |
| **STEP 5. 정규화** | 평균 0, 표준편차 1로 스케일 통일 | train 기준으로만 fit, test는 transform만 |
| **STEP 6. 함수로 묶기** | 파일 1개 처리 과정을 함수로 정리 | 재사용을 위한 함수화 |
| **STEP 7. 전체 파일로 확장** | 10개 파일 + 라벨 → 최종 데이터셋 완성 | concatenate로 합치고 X/y 만들기 |

```
.mat 파일
   ↓ STEP 1. 불러오기
원본 신호
   ↓ STEP 2. train / test 분리
   ↓ STEP 3. 윈도우 슬라이싱 (겹치게 자르기)
   ↓ STEP 4. FFT 변환
   ↓ STEP 5. 정규화
   ↓ STEP 6. 함수로 묶기
   ↓ STEP 7. 10개 파일 전체 적용 + 라벨링 + 합치기
딥러닝 모델 입력 데이터(X_train, X_test, y_train, y_test) 완성!
```
